## importing libraries

In [1]:
import pyspark

In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, column, count, when, split, expr
from pyspark.ml.feature import Imputer


In [3]:
ss=SparkSession.builder.master("local").appName("Crash Crew Data Cleaning").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/storage/home/mpf5654/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/storage/home/mpf5654/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
25/11/27 22:48:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/27 22:48:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
ss.sparkContext.setCheckpointDir("~/scratch")

## loading data

In [5]:
# Read in the ratings_2 file
df = ss.read.csv("sampled_accident_data.csv", header=True, inferSchema=True)
# In the cluster mode, we need to change to  `header=False` because the large file does not have header.

In [6]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

In [7]:
df.count()

77284

In [8]:
# Compute non-null counts for each column
non_null_counts = df.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df.columns
]).collect()[0]


# Print column name and count side by side
for c in df.columns:
    print(f"{c}: {non_null_counts[c]}")

25/11/27 22:48:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


ID: 77284
Source: 77284
Severity: 77284
Start_Time: 77284
End_Time: 77284
Start_Lat: 77284
Start_Lng: 77284
End_Lat: 43252
End_Lng: 43252
Distance(mi): 77284
Description: 77284
Street: 77158
City: 77282
County: 77284
State: 77284
Zipcode: 77263
Country: 77284
Timezone: 77201
Airport_Code: 77059
Weather_Timestamp: 76093
Temperature(F): 75687
Wind_Chill(F): 57352
Humidity(%): 75593
Pressure(in): 75879
Visibility(mi): 75516
Wind_Direction: 75528
Wind_Speed(mph): 71550
Precipitation(in): 55320
Weather_Condition: 75557
Amenity: 77284
Bump: 77284
Crossing: 77284
Give_Way: 77284
Junction: 77284
No_Exit: 77284
Railway: 77284
Roundabout: 77284
Station: 77284
Stop: 77284
Traffic_Calming: 77284
Traffic_Signal: 77284
Turning_Loop: 77284
Sunrise_Sunset: 77049
Civil_Twilight: 77049
Nautical_Twilight: 77049
Astronomical_Twilight: 77049


In [9]:
cols_to_impute = ["Temperature(F)", "Humidity(%)", "Visibility(mi)", "Precipitation(in)"]

imputer = Imputer(
    inputCols=cols_to_impute,
    outputCols=cols_to_impute
).setStrategy("mean")

df = imputer.fit(df).transform(df)

## Dropping null values

In [10]:
df_clean = df.na.drop(subset=cols_to_impute)

In [11]:
df_clean.count()

77284

Dropping rows with null values remove 53% of the data. Since our original data is around 7 million rows, removing null values is appropriate.

In [12]:
# count number of duplicates
total = df_clean.count()
unique = df_clean.dropDuplicates().count()
print(f"Total rows: {total}, Unique rows: {unique}")

Total rows: 77284, Unique rows: 77284


In [13]:
output_path = "cleaned_sampled_accident_data.csv"
df_clean.write.option("header", True).csv(output_path)

In [14]:
ss.stop()